# Import necessary libraries

In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np
import math
import uuid
import time
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from functools import lru_cache
from math import radians, sin, cos, asin, sqrt
from pathlib import Path
import os
import re
import logging

# Dynamic Routing

## 1. Utility functions: distance, normalization, time window

In [ ]:
from pandas.api.types import is_datetime64_any_dtype

def _round5(x):
    """Round to 1e-5° (~1m) and turn NaN into None (hashable for lru_cache)."""
    return None if pd.isna(x) else round(float(x), 5)

@lru_cache(maxsize=2_000_000)
def _hav_cached(lon1, lat1, lon2, lat2):
    if None in (lon1, lat1, lon2, lat2):
        return 0.0
    R = 6371.0
    dlon = radians(lon2 - lon1)
    dlat = radians(lat2 - lat1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2*R*asin(sqrt(a))

def haversine_km(lon1, lat1, lon2, lat2):
    return _hav_cached(_round5(lon1), _round5(lat1), _round5(lon2), _round5(lat2))

def normalize_requests_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ["pickup_datetime", "dropoff_datetime"]:
        if col in df.columns:
            if not is_datetime64_any_dtype(df[col]):
                df[col] = pd.to_datetime(
                    df[col],
                    errors="coerce",
                    format="mixed",
                )

            dtype = df[col].dtype
            if isinstance(dtype, pd.DatetimeTZDtype):
                df[col] = df[col].dt.tz_convert(None)

    if 'pickup_lon' not in df.columns and 'pickup_longitude' in df.columns:
        df['pickup_lon'] = df['pickup_longitude']
    if 'pickup_lat' not in df.columns and 'pickup_latitude' in df.columns:
        df['pickup_lat'] = df['pickup_latitude']
    if 'dropoff_lon' not in df.columns and 'dropoff_longitude' in df.columns:
        df['dropoff_lon'] = df['dropoff_longitude']
    if 'dropoff_lat' not in df.columns and 'dropoff_latitude' in df.columns:
        df['dropoff_lat'] = df['dropoff_latitude']

    if 'assigned' not in df.columns:
        df['assigned'] = False
    if 'assigned_taxi_id' not in df.columns:
        df['assigned_taxi_id'] = pd.Series([None] * len(df), dtype='object')
    if 'assigned_depot_id' not in df.columns:
        df['assigned_depot_id'] = pd.Series([None] * len(df), dtype='object')
    if 'assigned_mode' not in df.columns:
        df['assigned_mode'] = pd.Series([''] * len(df), dtype='object')

    return df

def ensure_pickup_windows(df: pd.DataFrame, early_s: int = 0, late_s: int = 180) -> pd.DataFrame:
    if 'pickup_earliest' not in df.columns:
        df['pickup_earliest'] = pd.to_datetime(df['pickup_datetime']) - pd.Timedelta(seconds=early_s)
    else:
        df['pickup_earliest'] = pd.to_datetime(df['pickup_earliest'])
    if 'pickup_latest' not in df.columns:
        df['pickup_latest']  = pd.to_datetime(df['pickup_datetime']) + pd.Timedelta(seconds=late_s)
    else:
        df['pickup_latest']  = pd.to_datetime(df['pickup_latest'])
    return df


def travel_minutes_km(km: float, speed_kmh: float = 25.0) -> float:
    return 0.0 if speed_kmh <= 0 else (km / speed_kmh) * 60.0


FARE_PER_HOUR_DRIVE   = 80.0   
COST_PER_HOUR_DRIVE   = 5.0   
COST_PER_HOUR_WAIT    = 1.0  

def compute_arc_profit_paper(
    prev_lon: float,
    prev_lat: float,
    prev_end_time, 
    req_row,             
    speed_kmh: float,
    fare_per_hour: float = FARE_PER_HOUR_DRIVE,
    cost_drive_per_hour: float = COST_PER_HOUR_DRIVE,
    cost_wait_per_hour: float = COST_PER_HOUR_WAIT,
):
    km_dead = haversine_km(prev_lon, prev_lat,
                           req_row["pickup_lon"], req_row["pickup_lat"])
    km_trip = haversine_km(req_row["pickup_lon"], req_row["pickup_lat"],
                           req_row["dropoff_lon"], req_row["dropoff_lat"])

    v = max(speed_kmh, 1e-9)
    t_dead_h = km_dead / v
    t_trip_h = km_trip / v

    arrival_time = prev_end_time + pd.to_timedelta(t_dead_h, unit="h")

    t_min_c = req_row["pickup_earliest"]
    wait_seconds = max(0.0, (t_min_c - arrival_time).total_seconds())
    wait_h = wait_seconds / 3600.0

    fare = fare_per_hour * t_trip_h
    cost_deadhead = cost_drive_per_hour * t_dead_h  
    cost_loaded = cost_drive_per_hour * t_trip_h   
    cost_drive_total = cost_deadhead + cost_loaded  
    cost_wait = cost_wait_per_hour * wait_h
    profit = fare - cost_drive_total - cost_wait

    return {
        "km_dead": km_dead,
        "km_trip": km_trip,
        "t_dead_h": t_dead_h,
        "t_trip_h": t_trip_h,
        "wait_h": wait_h,
        "fare_paper": fare,
        "cost_drive_paper": cost_drive_total,
        "cost_deadhead_paper": cost_deadhead,  
        "cost_loaded_paper": cost_loaded,     
        "cost_wait_paper": cost_wait,
        "profit_arc_paper": profit,
    }

def ok_small_pos_delta_km(delta_km: float, trip_km: float,
                          cap_km: float = 0.9, cap_frac: float = 0.30) -> bool:
    """Allow small positive delta_km (to make preempt/ejection less tight)"""
    if delta_km <= 1e-9:
        return True
    cap = min(float(cap_km), float(cap_frac) * max(float(trip_km), 1e-9))
    return delta_km <= cap + 1e-9


def taxi_remaining_frozen(tx: TaxiLive) -> int:

    return max(0, len(tx.route_freeze) - int(getattr(tx, "freeze_ptr", 0)))

def taxi_is_idle(tx: TaxiLive) -> bool:
    return (taxi_remaining_frozen(tx) == 0) and (len(tx.route_open) == 0)

def dynamic_candidate_limit(
    taxis_or_list,
    base_limit: int,
    min_limit: int = 20,
    include_near_idle: bool = True,
    near_idle_open_cap: int = 1
) -> int:
    if isinstance(taxis_or_list, dict):
        txs = list(taxis_or_list.values())
    else:
        txs = list(taxis_or_list)

    n_total = len(txs)
    if n_total == 0:
        return max(min_limit, base_limit)

    n_idle = sum(1 for tx in txs if taxi_is_idle(tx))

    if include_near_idle:
        n_near_idle = sum(
            1 for tx in txs
            if (taxi_remaining_frozen(tx) == 0 and len(tx.route_open) <= near_idle_open_cap)
        )
        n_available = max(n_idle, n_near_idle)
    else:
        n_available = n_idle

    lim = max(base_limit, min_limit, n_available)
    lim = min(lim, n_total)
    return int(lim)


## 2. Minimal static group bootstrap (necessary for dynamic simulation)

In [ ]:
@dataclass
class Taxi:
    taxi_id: str
    depot_id: str
    current_lon: float
    current_lat: float
    start_time: pd.Timestamp
    current_time: Optional[pd.Timestamp] = None

    def __post_init__(self):
        if self.current_time is None:
            self.current_time = self.start_time

@dataclass
class Fleet:
    taxis: List[Taxi]

    @classmethod
    def from_depots(cls, df: pd.DataFrame) -> "Fleet":
        if 'pickup_datetime' not in df.columns:
            raise ValueError("Missing 'pickup_datetime'.")
        start_time = pd.to_datetime(df['pickup_datetime']).min()
        if pd.isna(start_time):
            raise ValueError("Cannot determine start_time from pickup_datetime.")

        required = ['depot_id', 'depot_lon', 'depot_lat', 'taxis_per_depot']
        for c in required:
            if c not in df.columns:
                raise ValueError(f"Missing column '{c}' for Fleet construction.")

        depot_table = (
            df[required]
            .dropna(subset=['depot_id'])
            .drop_duplicates(subset=['depot_id'])
            .copy()
        )
        depot_table['taxis_per_depot'] = (
            pd.to_numeric(depot_table['taxis_per_depot'], errors='coerce')
              .fillna(0)
              .clip(lower=0)
              .round()
              .astype(int)
        )

        taxis: List[Taxi] = []
        for _, d in depot_table.iterrows():
            depot_id = str(d['depot_id'])
            depot_lon = float(d['depot_lon'])
            depot_lat = float(d['depot_lat'])
            n_taxis = int(d['taxis_per_depot'])
            for k in range(1, n_taxis + 1):
                taxis.append(Taxi(
                    taxi_id=f"{depot_id}-{k:03d}",
                    depot_id=depot_id,
                    current_lon=depot_lon,
                    current_lat=depot_lat,
                    start_time=start_time
                ))
        return cls(taxis=taxis)

## 3. TaxiLive status and build from Fleet

In [ ]:
@dataclass
class TaxiLive:
    taxi_id: str
    depot_id: str
    origin_lon: float
    origin_lat: float

    cur_lon: float
    cur_lat: float
    cur_time: pd.Timestamp

    route_freeze: List[int] = field(default_factory=list) 
    route_open:   List[int] = field(default_factory=list) 
                                                        
    freeze_ptr: int = 0
    reposition_km: float = 0.0


def build_taxis_live(df_out: pd.DataFrame, fleet: Fleet) -> Dict[str, TaxiLive]:
    """Initialize TaxiLive objects; origin from df_out's depot_map (fallback: fleet/current)."""
    start = pd.to_datetime(
        df_out['pickup_earliest'] if 'pickup_earliest' in df_out.columns else df_out['pickup_datetime']
    ).min()

    depot_map: Dict[str, Tuple[float, float]] = {}
    if {'depot_id', 'depot_lon', 'depot_lat'}.issubset(df_out.columns):
        dep = (df_out[['depot_id','depot_lon','depot_lat']]
               .dropna(subset=['depot_id'])
               .drop_duplicates('depot_id'))
        for _, r in dep.iterrows():
            depot_map[str(r['depot_id'])] = (float(r['depot_lon']), float(r['depot_lat']))

    taxis: Dict[str, TaxiLive] = {}
    for t in getattr(fleet, 'taxis', []):
        dep_id = getattr(t, 'depot_id', None)
        dlon = dlat = None
        if dep_id is not None and str(dep_id) in depot_map:
            dlon, dlat = depot_map[str(dep_id)]
        else:
            dlon = getattr(t, 'depot_lon', None)
            dlat = getattr(t, 'depot_lat', None)
        if dlon is None or dlat is None:
            dlon = getattr(t, 'current_lon', 0.0)
            dlat = getattr(t, 'current_lat', 0.0)

        taxis[t.taxi_id] = TaxiLive(
            taxi_id=t.taxi_id,
            depot_id=str(dep_id) if dep_id is not None else "",
            origin_lon=float(dlon),
            origin_lat=float(dlat),
            cur_lon=float(dlon),
            cur_lat=float(dlat),
            cur_time=getattr(t, 'current_time', start)
        )
    return taxis

## 4. Feasibility and cost

In [ ]:
def route_feasible_from_state(df: pd.DataFrame, tx: TaxiLive, freeze_seq: List[int], open_seq: List[int],
                              speed_kmh: float = 25.0, return_lateness: bool = False):
    cur_t = tx.cur_time
    cur_lon, cur_lat = tx.cur_lon, tx.cur_lat
    total_wait_s = 0.0

    for ridx in (freeze_seq + open_seq):
        r = df.loc[ridx]
        km_dead = haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
        eta = cur_t + pd.to_timedelta(travel_minutes_km(km_dead, speed_kmh), unit='m')
        px  = r['pickup_datetime']
        px_e = r['pickup_earliest'] if 'pickup_earliest' in df.columns else px
        px_l = r['pickup_latest']   if 'pickup_latest'   in df.columns else px

        if pd.notna(px_l) and eta > px_l:
            return (False, 0.0) if return_lateness else False
        if pd.notna(px_e):
            wait = (px_e - eta).total_seconds()
            if wait > 0:
                total_wait_s += wait

        cur_t = max(eta, px_e) if pd.notna(px_e) else eta

        km_load = haversine_km(r['pickup_lon'], r['pickup_lat'], r['dropoff_lon'], r['dropoff_lat'])
        cur_t = cur_t + pd.to_timedelta(travel_minutes_km(km_load, speed_kmh), unit='m')
        cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']

    return (True, total_wait_s) if return_lateness else True

def route_cost_km_current(df: pd.DataFrame, tx: TaxiLive, freeze_seq: List[int], open_seq: List[int]) -> float:
    cur_lon, cur_lat = tx.cur_lon, tx.cur_lat
    total = 0.0
    for ridx in (freeze_seq + open_seq):
        r = df.loc[ridx]
        total += haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
        total += haversine_km(r['pickup_lon'], r['pickup_lat'], r['dropoff_lon'], r['dropoff_lat'])
        cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']
    return total

def route_cost_km_from_origin(df: pd.DataFrame, seq: List[int], start_lon: float, start_lat: float) -> float:
    cur_lon, cur_lat = start_lon, start_lat
    total = 0.0
    for ridx in seq:
        r = df.loc[ridx]
        total += haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
        total += haversine_km(r['pickup_lon'], r['pickup_lat'], r['dropoff_lon'], r['dropoff_lat'])
        cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']
    return total

def deadhead_km_from_origin(df: pd.DataFrame, seq: List[int], start_lon: float, start_lat: float) -> float:
    cur_lon, cur_lat = start_lon, start_lat
    dead = 0.0
    for ridx in seq:
        r = df.loc[ridx]
        dead += haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
        cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']
    return dead

## 5. State Advance

In [ ]:
def update_freeze_open(df: pd.DataFrame, taxis: Dict[str, TaxiLive], now: pd.Timestamp, freeze_window_s: int):
    lock_until = now + pd.Timedelta(seconds=freeze_window_s)
    for tx in taxis.values():
        k = 0
        while k < len(tx.route_open):
            ridx = tx.route_open[k]
            px_e = df.at[ridx, 'pickup_earliest'] if 'pickup_earliest' in df.columns else df.at[ridx, 'pickup_datetime']
            if pd.isna(px_e) or px_e > lock_until:
                break
            k += 1
        if k > 0:
            tx.route_freeze.extend(tx.route_open[:k])
            tx.route_open = tx.route_open[k:]

def advance_state_after_freeze(df: pd.DataFrame, taxis: Dict[str, TaxiLive], speed_kmh: float):
    for tx in taxis.values():
        start_i = tx.freeze_ptr
        if start_i >= len(tx.route_freeze):
            continue

        cur_t  = tx.cur_time
        cur_lon, cur_lat = tx.cur_lon, tx.cur_lat

        for ridx in tx.route_freeze[start_i:]:
            r = df.loc[ridx]
            km_dead = haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
            eta = cur_t + pd.to_timedelta(travel_minutes_km(km_dead, speed_kmh), unit='m')
            px_e = r['pickup_earliest'] if 'pickup_earliest' in df.columns else r['pickup_datetime']

            cur_t = max(eta, px_e) if pd.notna(px_e) else eta
            if getattr(cur_t, "tzinfo", None) is not None:
                cur_t = cur_t.tz_convert(None)

            df.at[ridx, 'sim_pickup_datetime'] = cur_t

            km_load = haversine_km(r['pickup_lon'], r['pickup_lat'], r['dropoff_lon'], r['dropoff_lat'])
            cur_t = cur_t + pd.to_timedelta(travel_minutes_km(km_load, speed_kmh), unit='m')
            if getattr(cur_t, "tzinfo", None) is not None:
                cur_t = cur_t.tz_convert(None)

            df.at[ridx, 'sim_dropoff_datetime'] = cur_t

            cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']

        tx.cur_time, tx.cur_lon, tx.cur_lat = cur_t, cur_lon, cur_lat
        tx.freeze_ptr = len(tx.route_freeze)

def materialize_sim_times(df: pd.DataFrame, taxis: Dict[str, TaxiLive], speed_kmh: float):
    sim_dtype = "datetime64[ns]"
    if "sim_pickup_datetime" not in df.columns:
        df["sim_pickup_datetime"] = pd.Series(pd.NaT, index=df.index, dtype=sim_dtype)
    else:
        df["sim_pickup_datetime"] = pd.to_datetime(df["sim_pickup_datetime"], errors="coerce")
        try:
            df["sim_pickup_datetime"] = df["sim_pickup_datetime"].dt.tz_localize(None)
        except TypeError:
            pass

    if "sim_dropoff_datetime" not in df.columns:
        df["sim_dropoff_datetime"] = pd.Series(pd.NaT, index=df.index, dtype=sim_dtype)
    else:
        df["sim_dropoff_datetime"] = pd.to_datetime(df["sim_dropoff_datetime"], errors="coerce")
        try:
            df["sim_dropoff_datetime"] = df["sim_dropoff_datetime"].dt.tz_localize(None)
        except TypeError:
            pass

    to_minutes = lambda km: pd.to_timedelta(travel_minutes_km(km, speed_kmh), unit='m')

    for tx in taxis.values():
        seq = tx.route_open
        cur_t  = tx.cur_time
        cur_lon, cur_lat = tx.cur_lon, tx.cur_lat

        for ridx in seq:
            r = df.loc[ridx]
            km_dead = haversine_km(cur_lon, cur_lat, r['pickup_lon'], r['pickup_lat'])
            eta = cur_t + to_minutes(km_dead)
            px_e = r['pickup_earliest'] if 'pickup_earliest' in df.columns else r['pickup_datetime']

            cur_t = max(eta, px_e) if pd.notna(px_e) else eta
            if getattr(cur_t, "tzinfo", None) is not None:
                cur_t = cur_t.tz_convert(None)

            df.at[ridx, 'sim_pickup_datetime'] = cur_t

            km_load = haversine_km(r['pickup_lon'], r['pickup_lat'], r['dropoff_lon'], r['dropoff_lat'])
            cur_t = cur_t + to_minutes(km_load)

            if getattr(cur_t, "tzinfo", None) is not None:
                cur_t = cur_t.tz_convert(None)
            df.at[ridx, 'sim_dropoff_datetime'] = cur_t

            cur_lon, cur_lat = r['dropoff_lon'], r['dropoff_lat']


def compute_profit_paper_style(
    df: pd.DataFrame,
    taxis_live: dict,
    speed_kmh: float,
    fare_per_hour: float = FARE_PER_HOUR_DRIVE,
    cost_drive_per_hour: float = COST_PER_HOUR_DRIVE,
    cost_wait_per_hour: float = COST_PER_HOUR_WAIT,
):
    df = df.copy()

    mask = (df.get("assigned", True) == True) & df["sim_pickup_datetime"].notna()
    served = df[mask]

    profit_by_taxi = {}
    system_profit = 0.0
    system_cost_deadhead = 0.0  
    system_cost_loaded = 0.0    
    system_fare = 0.0
    system_cost_drive = 0.0
    system_cost_wait = 0.0

    for tid, grp in served.groupby("assigned_taxi_id"):
        if pd.isna(tid):
            continue

        grp = grp.sort_values("sim_pickup_datetime").copy()

        tx = taxis_live[str(tid)] if isinstance(taxis_live, dict) else taxis_live[tid]

        prev_lon = getattr(tx, "origin_lon", getattr(tx, "cur_lon", 0.0))
        prev_lat = getattr(tx, "origin_lat", getattr(tx, "cur_lat", 0.0))

        start_time = pd.to_datetime(grp["pickup_earliest"]).min()
        last_dropoff_time = start_time

        total_fare = 0.0
        total_cost_drive = 0.0
        total_cost_deadhead = 0.0 
        total_cost_loaded = 0.0   
        total_cost_wait = 0.0
        total_profit = 0.0

        for _, r in grp.iterrows():
            prev_time = last_dropoff_time

            res = compute_arc_profit_paper(
                prev_lon=prev_lon,
                prev_lat=prev_lat,
                prev_end_time=prev_time,
                req_row=r,
                speed_kmh=speed_kmh,
                fare_per_hour=fare_per_hour,
                cost_drive_per_hour=cost_drive_per_hour,
                cost_wait_per_hour=cost_wait_per_hour,
            )

            idx = r.name 
            df.at[idx, 'km_dead_paper'] = res['km_dead']
            df.at[idx, 'km_trip_paper'] = res['km_trip']
            df.at[idx, 'km_total_paper'] = res['km_dead'] + res['km_trip']

            last_dropoff_time = r["sim_dropoff_datetime"]

            total_fare += res["fare_paper"]
            total_cost_drive += res["cost_drive_paper"]
            total_cost_deadhead += res["cost_deadhead_paper"]
            total_cost_loaded += res["cost_loaded_paper"]
            total_cost_wait += res["cost_wait_paper"]
            total_profit += res["profit_arc_paper"]

        profit_by_taxi[str(tid)] = {
            "total_fare_paper": total_fare,
            "total_cost_drive_paper": total_cost_drive,
            "total_cost_deadhead_paper": total_cost_deadhead,
            "total_cost_loaded_paper": total_cost_loaded,
            "total_cost_wait_paper": total_cost_wait,
            "total_profit_paper": total_profit,
        }

        system_profit += total_profit
        system_fare += total_fare
        system_cost_drive += total_cost_drive
        system_cost_deadhead += total_cost_deadhead
        system_cost_loaded += total_cost_loaded
        system_cost_wait += total_cost_wait

    return df, profit_by_taxi, system_profit, system_fare, system_cost_drive, system_cost_wait, system_cost_deadhead, system_cost_loaded


def print_top5_taxis_profit_from_df(
    df: pd.DataFrame,
    taxis_live: Dict[str, TaxiLive],
    speed_kmh: float,
    top_n: int = 5,
):
    df_profit, profit_by_taxi, system_profit = compute_profit_paper_style(
        df,
        taxis_live,
        speed_kmh=speed_kmh,
    )

    print(f"\nTotal system profit (paper-style): {system_profit:.2f}")

    rows = []
    for tid, stats in profit_by_taxi.items():
        rows.append({
            "taxi_id": tid,
            "total_profit": float(stats.get("total_profit_paper", 0.0)),
            "total_fare": float(stats.get("total_fare_paper", 0.0)),
            "total_cost_drive": float(stats.get("total_cost_drive_paper", 0.0)),
            "total_cost_wait": float(stats.get("total_cost_wait_paper", 0.0)),
        })

    dfp = pd.DataFrame(rows)
    if dfp.empty:
        print("No taxis have profit (profit_by_taxi is empty).")
        return

    dfp_sorted = dfp.sort_values("total_profit", ascending=False)
    top_n = min(top_n, len(dfp_sorted))

    print(f"\n=== Top {top_n} taxis with the most LOTS ===")
    print(dfp_sorted.head(top_n).to_string(index=False))

    print(f"\n=== Top {top_n} taxis with the most LOSS ===")
    print(
        dfp_sorted.tail(top_n)
        .sort_values("total_profit")
        .to_string(index=False)
    )


## 6. Candidate evaluation and insertion functions

In [ ]:
def _eta_minutes_to_point(tx: TaxiLive, lon: float, lat: float, speed_kmh: float) -> float:
    km = haversine_km(tx.cur_lon, tx.cur_lat, lon, lat)
    return travel_minutes_km(km, speed_kmh)

def nearest_taxis_for_point_eta(taxis: Dict[str, TaxiLive], lon: float, lat: float, k: int, speed_kmh: float) -> List[TaxiLive]:
    lst = list(taxis.values())
    lst.sort(key=lambda tx: _eta_minutes_to_point(tx, lon, lat, speed_kmh))
    return lst[:max(1, min(k, len(lst)))]

def taxi_pending_load(tx: TaxiLive) -> int:
    pending_freeze = max(0, len(tx.route_freeze) - int(getattr(tx, "freeze_ptr", 0)))
    return int(pending_freeze + len(tx.route_open))

def try_insert_best_with_load_bias(
    df: pd.DataFrame,
    taxis: Dict[str, TaxiLive],
    ridx: int,
    speed_kmh: float = 25.0,
    candidate_taxi_limit: int = 100,
    pos_limit: int = 15,
    depot_bonus: float = 0.5,    
    idle_bonus: float = 0.3,        
    max_detour_ratio: float = 1.5 
) -> bool:
    row = df.loc[ridx]
    px_lon, px_lat = row['pickup_lon'], row['pickup_lat']
    dx_lon, dx_lat = row['dropoff_lon'], row['dropoff_lat']

    trip_distance = haversine_km(px_lon, px_lat, dx_lon, dx_lat)

    def _is_at_depot(tx: TaxiLive) -> bool:
        dist_to_depot = haversine_km(tx.cur_lon, tx.cur_lat, tx.origin_lon, tx.origin_lat)
        return dist_to_depot <= 0.1


    cand_limit = dynamic_candidate_limit(taxis, candidate_taxi_limit, min_limit=30, include_near_idle=True)
    cand_taxis = nearest_taxis_for_point_eta(taxis, px_lon, px_lat, cand_limit, speed_kmh)


    best_overall = None
    best_key = None  

    for tx in cand_taxis:
        m = len(tx.route_open)

        head = list(range(0, min(pos_limit, m) + 1))
        tail = list(range(max(0, m - pos_limit), m + 1))
        positions = sorted(set(head + tail)) if m > pos_limit else list(range(0, m + 1))

        base_km = route_cost_km_current(df, tx, [], tx.route_open)

        for pos in positions:
            cand_open = tx.route_open[:pos] + [ridx] + tx.route_open[pos:]
            ok, wait_s = route_feasible_from_state(df, tx, [], cand_open, speed_kmh, return_lateness=True)
            if not ok:
                continue

            new_km = route_cost_km_current(df, tx, [], cand_open)
            delta_km = new_km - base_km

            max_ratio = max_detour_ratio
            if delta_km > 0 and trip_distance > 0:
                detour_ratio = (delta_km / trip_distance)
                if detour_ratio > max_ratio:
                    continue

            eta_minutes = _eta_minutes_to_point(tx, px_lon, px_lat, speed_kmh)
            key = (float(eta_minutes), float(wait_s), float(delta_km))
            if (best_key is None) or (key < best_key):
                best_key = key
                best_overall = (tx, pos, delta_km, eta_minutes, wait_s)

    if best_overall is None:
        return False

    tx, pos, delta_km, eta_min, wait_s = best_overall
    tx.route_open = tx.route_open[:pos] + [ridx] + tx.route_open[pos:]

    df.at[ridx, 'assigned'] = True
    df.at[ridx, 'assigned_taxi_id'] = tx.taxi_id
    df.at[ridx, 'assigned_depot_id'] = tx.depot_id

    df.at[ridx, 'assigned_mode'] = ('insert_from_depot' if _is_at_depot(tx) else 'insert_near')
    return True

def preempt_if_better(
    df: pd.DataFrame, taxis: Dict[str, TaxiLive], ridx_new: int, speed_kmh: float = 25.0,
    candidate_taxi_limit: int = 20, preempt_positions: int = 3
) -> bool:
    ALLOW_SMALL_POS_DELTA = True
    POS_DELTA_CAP_KM = 0.8
    POS_DELTA_FRAC_OF_TRIP = 0.25

    row = df.loc[ridx_new]
    px_lon, px_lat = row['pickup_lon'], row['pickup_lat']

    trip_km_new = haversine_km(row['pickup_lon'], row['pickup_lat'],
                              row['dropoff_lon'], row['dropoff_lat'])

    def _ok_small_positive(delta: float) -> bool:
        if not ALLOW_SMALL_POS_DELTA:
            return False
        cap = min(float(POS_DELTA_CAP_KM), float(POS_DELTA_FRAC_OF_TRIP) * max(trip_km_new, 1e-9))
        return delta <= cap + 1e-9

    cand_limit = dynamic_candidate_limit(taxis, candidate_taxi_limit, min_limit=25, include_near_idle=True)
    cand_taxis = nearest_taxis_for_point_eta(taxis, px_lon, px_lat, cand_limit, speed_kmh)

    best: Optional[Tuple[tuple, str, int, float]] = None 

    for tx in cand_taxis:
        m = len(tx.route_open)
        lim = min(preempt_positions, m)
        base = route_cost_km_current(df, tx, [], tx.route_open)

        for pos in range(0, lim + 1):
            cand_open = tx.route_open[:pos] + [ridx_new] + tx.route_open[pos:]
            ok, wait_s = route_feasible_from_state(df, tx, [], cand_open, speed_kmh, return_lateness=True)
            if not ok:
                continue

            newc = route_cost_km_current(df, tx, [], cand_open)
            delta = newc - base

            if m > 0 and delta > 1e-9 and not _ok_small_positive(delta):
                continue

            eta_min = _eta_minutes_to_point(tx, px_lon, px_lat, speed_kmh)
            key = (int(pos), float(wait_s), float(eta_min))
            if (best is None) or (key < best[0]):
                best = (key, tx.taxi_id, pos, wait_s)

    if best is None:
        return False

    _, tid, pos, _ = best
    tx = taxis[tid]
    tx.route_open = tx.route_open[:pos] + [ridx_new] + tx.route_open[pos:]
    df.at[ridx_new, 'assigned'] = True
    df.at[ridx_new, 'assigned_taxi_id'] = tid
    df.at[ridx_new, 'assigned_depot_id'] = tx.depot_id
    df.at[ridx_new, 'assigned_mode'] = ('preempt_front' if pos == 0 else f'preempt_pos{pos}')
    return True


## 7. Local serch functions on the open paragraph (changes are allowed)

In [ ]:
class MoveStats:
    def __init__(self):
        self.count = {'relocate':0, 'swap':0, 'cross':0}
        self.delta = {'relocate':0.0, 'swap':0.0, 'cross':0.0}


@dataclass(frozen=True)
class MoveProposal:
    name: str
    delta: float
    payload: tuple


def apply_move(proposal: MoveProposal, taxis: Dict[str, TaxiLive]) -> bool:
    name = proposal.name

    if name == 'relocate':
        a, i, b, j, ridx = proposal.payload
        if a not in taxis or b not in taxis:
            return False
        ta, tb = taxis[a], taxis[b]

        if i < len(ta.route_open) and ta.route_open[i] == ridx:
            ta.route_open.pop(i)
        else:
            try:
                ta.route_open.remove(ridx)
            except ValueError:
                return False

        if a == b:
            if j > i:
                j = j - 1
            tb = ta

        j = max(0, min(int(j), len(tb.route_open)))
        tb.route_open = tb.route_open[:j] + [ridx] + tb.route_open[j:]
        return True

    if name == 'swap':
        a, i, b, j, ra, rb = proposal.payload
        if a not in taxis or b not in taxis:
            return False
        ta, tb = taxis[a], taxis[b]

        if not (0 <= i < len(ta.route_open)) or ta.route_open[i] != ra:
            try:
                i = ta.route_open.index(ra)
            except ValueError:
                return False
        if not (0 <= j < len(tb.route_open)) or tb.route_open[j] != rb:
            try:
                j = tb.route_open.index(rb)
            except ValueError:
                return False

        ta.route_open[i], tb.route_open[j] = tb.route_open[j], ta.route_open[i]
        return True

    if name == 'cross':
        a, i, b, j, k, Ablock, Bblock = proposal.payload
        if a not in taxis or b not in taxis:
            return False
        ta, tb = taxis[a], taxis[b]

        k = int(k)
        if k <= 0:
            return False
        if not (0 <= i <= len(ta.route_open) - k) or not (0 <= j <= len(tb.route_open) - k):
            return False

        if ta.route_open[i:i+k] != list(Ablock):
            try:
                i = ta.route_open.index(Ablock[0])
            except Exception:
                return False
            if ta.route_open[i:i+k] != list(Ablock):
                return False

        if tb.route_open[j:j+k] != list(Bblock):
            try:
                j = tb.route_open.index(Bblock[0])
            except Exception:
                return False
            if tb.route_open[j:j+k] != list(Bblock):
                return False

        ta.route_open = ta.route_open[:i] + list(Bblock) + ta.route_open[i+k:]
        tb.route_open = tb.route_open[:j] + list(Ablock) + tb.route_open[j+k:]
        return True

    return False


def relocate_best_improvement(
    df: pd.DataFrame, taxis: Dict[str, TaxiLive], speed_kmh: float = 25.0,
    candidate_taxi_limit: int = 20, pos_limit: int = 6
) -> Optional[MoveProposal]:
    import random
    best: Optional[Tuple[float, tuple]] = None

    tids = list(taxis.keys())
    random.shuffle(tids)

    for a in tids:
        ta = taxis[a]
        ma = len(ta.route_open)
        if ma == 0:
            continue

        head_a = list(range(0, min(pos_limit, ma)))
        tail_a = list(range(max(0, ma - pos_limit), ma))
        idx_a = sorted(set(head_a + tail_a))

        base_a = route_cost_km_current(df, ta, [], ta.route_open)

        for i in idx_a:
            ridx = ta.route_open[i]
            lon_i, lat_i = df.at[ridx,'pickup_lon'], df.at[ridx,'pickup_lat']
            near_b = nearest_taxis_for_point_eta(taxis, lon_i, lat_i, candidate_taxi_limit, speed_kmh)

            for tb in near_b:
                b = tb.taxi_id
                mb = len(tb.route_open)

                head_b = list(range(0, min(pos_limit, mb)+1))
                tail_b = list(range(max(0, mb - pos_limit), mb+1))
                pos_b = sorted(set(head_b + tail_b))

                base_b = route_cost_km_current(df, tb, [], tb.route_open)

                for j in pos_b:
                    if a == b:
                        if j == i or j == i + 1:
                            continue
                        new_open = ta.route_open.copy()
                        item = new_open.pop(i)
                        j_adj = j - 1 if j > i else j
                        j_adj = max(0, min(j_adj, len(new_open)))
                        new_open.insert(j_adj, item)

                        if not route_feasible_from_state(df, ta, [], new_open, speed_kmh):
                            continue
                        newc = route_cost_km_current(df, ta, [], new_open)
                        delta = newc - base_a
                        if delta < -1e-9 and (best is None or delta < best[0]):
                            best = (delta, (a, i, b, j, ridx))
                        continue

                    if mb == 0:
                        pass

                    base = base_a + base_b
                    new_a_open = ta.route_open[:i] + ta.route_open[i+1:]
                    new_b_open = tb.route_open[:j] + [ridx] + tb.route_open[j:]

                    if not route_feasible_from_state(df, ta, [], new_a_open, speed_kmh):
                        continue
                    if not route_feasible_from_state(df, tb, [], new_b_open, speed_kmh):
                        continue

                    newc = route_cost_km_current(df, ta, [], new_a_open) + route_cost_km_current(df, tb, [], new_b_open)
                    delta = newc - base
                    if delta < -1e-9 and (best is None or delta < best[0]):
                        best = (delta, (a, i, b, j, ridx))

    if best is None:
        return None
    delta, payload = best
    return MoveProposal(name='relocate', delta=float(delta), payload=payload)


def swap_best_improvement(
    df: pd.DataFrame, taxis: Dict[str, TaxiLive], speed_kmh: float = 25.0,
    candidate_taxi_limit: int = 20, pos_limit: int = 6
) -> Optional[MoveProposal]:
    best: Optional[Tuple[float, tuple]] = None

    tids = list(taxis.keys())
    for a in tids:
        ta = taxis[a]
        ma = len(ta.route_open)
        if ma == 0:
            continue

        head_a = list(range(0, min(pos_limit, ma)))
        tail_a = list(range(max(0, ma - pos_limit), ma))
        idx_a = sorted(set(head_a + tail_a))

        base_a = route_cost_km_current(df, ta, [], ta.route_open)

        for i in idx_a:
            ra = ta.route_open[i]
            lon_i, lat_i = df.at[ra,'pickup_lon'], df.at[ra,'pickup_lat']
            near_b = nearest_taxis_for_point_eta(taxis, lon_i, lat_i, candidate_taxi_limit, speed_kmh)

            for tb in near_b:
                b = tb.taxi_id
                mb = len(tb.route_open)
                if mb == 0:
                    continue

                head_b = list(range(0, min(pos_limit, mb)))
                tail_b = list(range(max(0, mb - pos_limit), mb))
                idx_b = sorted(set(head_b + tail_b))

                base_b = route_cost_km_current(df, tb, [], tb.route_open)

                for j in idx_b:
                    if a == b and i == j:
                        continue

                    rb = tb.route_open[j]

                    if a == b:
                        new_open = ta.route_open.copy()
                        new_open[i], new_open[j] = new_open[j], new_open[i]
                        if not route_feasible_from_state(df, ta, [], new_open, speed_kmh):
                            continue
                        newc = route_cost_km_current(df, ta, [], new_open)
                        delta = newc - base_a
                        if delta < -1e-9 and (best is None or delta < best[0]):
                            best = (delta, (a, i, b, j, ra, rb))
                        continue

                    base = base_a + base_b
                    new_a = ta.route_open.copy()
                    new_b = tb.route_open.copy()
                    new_a[i], new_b[j] = new_b[j], new_a[i]

                    if not route_feasible_from_state(df, ta, [], new_a, speed_kmh):
                        continue
                    if not route_feasible_from_state(df, tb, [], new_b, speed_kmh):
                        continue

                    newc = route_cost_km_current(df, ta, [], new_a) + route_cost_km_current(df, tb, [], new_b)
                    delta = newc - base
                    if delta < -1e-9 and (best is None or delta < best[0]):
                        best = (delta, (a, i, b, j, ra, rb))

    if best is None:
        return None
    delta, payload = best
    return MoveProposal(name='swap', delta=float(delta), payload=payload)


def cross_exchange_best_improvement(
    df: pd.DataFrame, taxis: Dict[str, TaxiLive], k: int = 1, speed_kmh: float = 25.0,
    candidate_taxi_limit: int = 20, pos_limit: int = 6
) -> Optional[MoveProposal]:
    best: Optional[Tuple[float, tuple]] = None

    tids = list(taxis.keys())
    k = int(k)

    for a in tids:
        ta = taxis[a]
        ma = len(ta.route_open)
        if ma < k:
            continue

        i_max = max(1, min(pos_limit, ma - k + 1))
        for i in range(0, i_max):
            Ablock = ta.route_open[i:i+k]
            ridx0 = Ablock[0]
            lon0, lat0 = df.at[ridx0,'pickup_lon'], df.at[ridx0,'pickup_lat']
            near_b = nearest_taxis_for_point_eta(taxis, lon0, lat0, candidate_taxi_limit, speed_kmh)

            for tb in near_b:
                b = tb.taxi_id

                if a == b:
                    continue

                mb = len(tb.route_open)
                if mb < k:
                    continue

                j_max = max(1, min(pos_limit, mb - k + 1))
                for j in range(0, j_max):
                    Bblock = tb.route_open[j:j+k]

                    new_a = ta.route_open[:i] + Bblock + ta.route_open[i+k:]
                    new_b = tb.route_open[:j] + Ablock + tb.route_open[j+k:]

                    if not route_feasible_from_state(df, ta, [], new_a, speed_kmh):
                        continue
                    if not route_feasible_from_state(df, tb, [], new_b, speed_kmh):
                        continue

                    base = (route_cost_km_current(df, ta, [], ta.route_open)
                            + route_cost_km_current(df, tb, [], tb.route_open))
                    newc = (route_cost_km_current(df, ta, [], new_a)
                            + route_cost_km_current(df, tb, [], new_b))
                    delta = newc - base
                    if delta < -1e-9 and (best is None or delta < best[0]):
                        best = (delta, (a, i, b, j, k, list(Ablock), list(Bblock)))

    if best is None:
        return None
    delta, payload = best
    return MoveProposal(name='cross', delta=float(delta), payload=payload)

def cap_open_lists(df: pd.DataFrame, taxis: Dict[str, TaxiLive], m_max_open: int):
    if not m_max_open or m_max_open <= 0:
        return []

    removed_total = []
    for tx in taxis.values():
        m = len(tx.route_open)
        if m <= m_max_open:
            continue

        key_ts = (lambda ridx: df.at[ridx, 'pickup_earliest'] if 'pickup_earliest' in df.columns
                  else df.at[ridx, 'pickup_datetime'])
        idx_sorted_desc = sorted(range(m), key=lambda i: key_ts(tx.route_open[i]), reverse=True)
        remove_idx = set(idx_sorted_desc[: m - m_max_open])

        removed = [tx.route_open[i] for i in sorted(remove_idx)]
        for ridx in removed:
            df.at[ridx, 'assigned'] = False
            df.at[ridx, 'assigned_taxi_id'] = None
            df.at[ridx, 'assigned_depot_id'] = None
            df.at[ridx, 'assigned_mode'] = 'deferred_by_cap'

        tx.route_open = [r for i, r in enumerate(tx.route_open) if i not in remove_idx]
        removed_total.extend(removed)

    return removed_total

## 8.

In [ ]:
def assign_with_ejection(
    df: pd.DataFrame, taxis: Dict[str, TaxiLive],
    ridx_new: int, speed_kmh: float = 25.0,
    candidate_taxi_limit: int = 50, pos_limit: int = 12,
    eject_window: int = 2
) -> bool:
    row = df.loc[ridx_new]
    px_lon, px_lat = row['pickup_lon'], row['pickup_lat']

    trip_km_new = haversine_km(row['pickup_lon'], row['pickup_lat'],
                              row['dropoff_lon'], row['dropoff_lat'])

    cand_limit = dynamic_candidate_limit(taxis, candidate_taxi_limit, min_limit=35, include_near_idle=True)
    cand_taxis = nearest_taxis_for_point_eta(taxis, px_lon, px_lat, cand_limit, speed_kmh)


    for ta in cand_taxis:
        m = len(ta.route_open)
        head = list(range(0, min(pos_limit, m) + 1))
        tail = list(range(max(0, m - pos_limit), m + 1))
        positions = sorted(set(head + tail)) if m > pos_limit else list(range(0, m + 1))

        baseA0 = route_cost_km_current(df, ta, [], ta.route_open)

        for pos in positions:
            cand_open = ta.route_open[:pos] + [ridx_new] + ta.route_open[pos:]
            ok, _late = route_feasible_from_state(df, ta, [], cand_open, speed_kmh, return_lateness=True)
            if ok:
                newA0 = route_cost_km_current(df, ta, [], cand_open)
                deltaA0 = newA0 - baseA0

                if not ok_small_pos_delta_km(deltaA0, trip_km_new, cap_km=0.9, cap_frac=0.30):
                    continue

                ta.route_open = cand_open
                df.at[ridx_new, 'assigned'] = True
                df.at[ridx_new, 'assigned_taxi_id'] = ta.taxi_id
                df.at[ridx_new, 'assigned_depot_id'] = ta.depot_id
                df.at[ridx_new, 'assigned_mode'] = 'insert_open_eject0'
                return True

            if m == 0:
                continue

            k_low = max(0, pos - eject_window)
            k_high = min(m - 1, pos + max(0, eject_window - 1))
            for k in range(k_low, k_high + 1):
                ridx_k = ta.route_open[k]
                open_wo_k = ta.route_open[:k] + ta.route_open[k + 1:]
                pos_after = pos if k >= pos else max(0, pos - 1)
                open_newA = open_wo_k[:pos_after] + [ridx_new] + open_wo_k[pos_after:]

                okA, _ = route_feasible_from_state(df, ta, [], open_newA, speed_kmh, return_lateness=True)
                if not okA:
                    continue

                newA = route_cost_km_current(df, ta, [], open_newA)
                deltaA = newA - baseA0

                if not ok_small_pos_delta_km(deltaA, trip_km_new, cap_km=0.9, cap_frac=0.30):
                    continue

                row_k = df.loc[ridx_k]
                k_px_lon, k_px_lat = row_k['pickup_lon'], row_k['pickup_lat']
                trip_km_k = haversine_km(row_k['pickup_lon'], row_k['pickup_lat'],
                                        row_k['dropoff_lon'], row_k['dropoff_lat'])

                target_taxis = [txb for txb in taxis.values() if txb.taxi_id != ta.taxi_id]
                target_taxis.sort(key=lambda txb: _eta_minutes_to_point(txb, k_px_lon, k_px_lat, speed_kmh))

                limitB = dynamic_candidate_limit(target_taxis, candidate_taxi_limit, min_limit=35, include_near_idle=True)
                target_taxis = target_taxis[:limitB]

                for tb in target_taxis:
                    m2 = len(tb.route_open)
                    baseB = route_cost_km_current(df, tb, [], tb.route_open)

                    head2 = list(range(0, min(pos_limit, m2) + 1))
                    tail2 = list(range(max(0, m2 - pos_limit), m2 + 1))
                    positions2 = sorted(set(head2 + tail2)) if m2 > pos_limit else list(range(0, m2 + 1))

                    for pos2 in positions2:
                        cand_open2 = tb.route_open[:pos2] + [ridx_k] + tb.route_open[pos2:]
                        okB, lateB = route_feasible_from_state(df, tb, [], cand_open2, speed_kmh, return_lateness=True)
                        if not okB:
                            continue

                        newcB = route_cost_km_current(df, tb, [], cand_open2)
                        deltaB = newcB - baseB

                        if not ok_small_pos_delta_km(deltaB, trip_km_k, cap_km=0.9, cap_frac=0.30):
                            continue

                        tb.route_open = cand_open2
                        df.at[ridx_k, 'assigned'] = True
                        df.at[ridx_k, 'assigned_taxi_id'] = tb.taxi_id
                        df.at[ridx_k, 'assigned_depot_id'] = tb.depot_id
                        df.at[ridx_k, 'assigned_mode'] = 'reinsert_after_eject'

                        ta.route_open = open_newA
                        df.at[ridx_new, 'assigned'] = True
                        df.at[ridx_new, 'assigned_taxi_id'] = ta.taxi_id
                        df.at[ridx_new, 'assigned_depot_id'] = ta.depot_id
                        df.at[ridx_new, 'assigned_mode'] = f'insert_open_eject1(k={ridx_k})'
                        return True

    return False


## 9. Pull new request every tick

In [ ]:
def pull_new_requests(df: pd.DataFrame, last_now: pd.Timestamp, now: pd.Timestamp) -> List[int]:
    """
    Retry ALL unassigned & unexpired requests.
    """
    base_mask = (
        (df["assigned"] == False)
        & (df["assigned_mode"] != "rejected")
        & (df["pickup_earliest"] <= now)
    )

    has_latest = "pickup_latest" in df.columns
    if has_latest:
        not_expired = (df["pickup_latest"].isna()) | (df["pickup_latest"] >= now)
    else:
        not_expired = pd.Series(True, index=df.index)

    mask_pending = base_mask & not_expired
    if not mask_pending.any():
        return []

    if has_latest:
        far_future = now + pd.Timedelta(days=3650)
        order = df.loc[mask_pending, ["pickup_latest", "pickup_earliest"]].copy()
        order["pickup_latest"] = pd.to_datetime(order["pickup_latest"]).fillna(far_future)
        order["pickup_earliest"] = pd.to_datetime(order["pickup_earliest"])
        idx = order.sort_values(["pickup_latest", "pickup_earliest"]).index.tolist()
    else:
        order = df.loc[mask_pending, ["pickup_earliest"]].copy()
        idx = order.sort_values(["pickup_earliest"]).index.tolist()

    return idx

## 10. The act of returning to the depot when free

In [ ]:
def reposition_idle_taxis_to_depot(taxis: Dict[str, TaxiLive], now: pd.Timestamp,
                                   speed_kmh: float, eps_km: float = 0.05) -> None:
    """
    For each taxi that is free (no longer route_freeze/open), move it to the depot (origin)
    with speed_kmh. Update current location & time to 'now' (no virtual request).
    eps_km: threshold considered to be at the depot.
    """
    if speed_kmh <= 0:
        return

    for tx in taxis.values():
        is_freeze_done = (tx.freeze_ptr == len(tx.route_freeze))
        if not (is_freeze_done and len(tx.route_open) == 0):
            continue

        if tx.cur_time >= now:
            continue

        allow_min = (now - tx.cur_time).total_seconds() / 60.0
        if allow_min <= 0:
            continue

        dist_km = haversine_km(tx.cur_lon, tx.cur_lat, tx.origin_lon, tx.origin_lat)

        if dist_km <= eps_km:
            tx.cur_lon, tx.cur_lat = tx.origin_lon, tx.origin_lat
            tx.cur_time = now
            continue

        move_km = speed_kmh * (allow_min / 60.0)

        moved_km = min(move_km, dist_km)
        tx.reposition_km += moved_km

        if move_km >= dist_km - 1e-9:
            travel_min = 60.0 * dist_km / speed_kmh
            tx.cur_lon, tx.cur_lat = tx.origin_lon, tx.origin_lat
            tx.cur_time = now
        else:
            ratio = move_km / max(dist_km, 1e-12)
            tx.cur_lon = tx.cur_lon + ratio * (tx.origin_lon - tx.cur_lon)
            tx.cur_lat = tx.cur_lat + ratio * (tx.origin_lat - tx.cur_lat)
            tx.cur_time = tx.cur_time + pd.to_timedelta(allow_min, unit='m')

## 11. New behavior - move to potential zone when free

### 11.1 Data path

In [ ]:
DATA_PATH = Path(r'd:/DTRP-DP/Routing/Input')

### 11.2 Load dataframes read from file,...

In [ ]:
class ZoneCentroids:
    def __init__(self, zone_df: pd.DataFrame):
        zdf = zone_df.copy()
        if 'zone_id' not in zdf.columns and 'zone' in zdf.columns:
            zdf = zdf.rename(columns={'zone': 'zone_id'})
        if 'longitude' not in zdf.columns:
            for alt in ['lon', 'long', 'x']:
                if alt in zdf.columns:
                    zdf = zdf.rename(columns={alt: 'longitude'})
                    break
        if 'latitude' not in zdf.columns:
            for alt in ['lat', 'y']:
                if alt in zdf.columns:
                    zdf = zdf.rename(columns={alt: 'latitude'})
                    break
        zdf['zone_id'] = zdf['zone_id'].astype(str)
        self._centroids: Dict[str, Tuple[float, float]] = {
            str(r['zone_id']): (float(r['longitude']), float(r['latitude']))
            for _, r in zdf.iterrows()
        }
    def centroid(self, zone_id: str) -> Tuple[float, float]:
        return self._centroids[str(zone_id)]
    @property
    def zone_ids(self) -> List[str]:
        return list(self._centroids.keys())

class ForecastAccessor:
    def __init__(self, forecast_df: pd.DataFrame, slot_minutes: int = 15):
        df = forecast_df.copy()
        if 'zone_id' not in df.columns and 'zone' in df.columns:
            df = df.rename(columns={'zone': 'zone_id'})
        if 'pred_count' not in df.columns:
            for alt in ['pred', 'count', 'yhat', 'forecast', 'value']:
                if alt in df.columns:
                    df = df.rename(columns={alt: 'pred_count'})
                    break
        if 'slot_start' not in df.columns:
            for alt in ['slot', 'time', 'ts', 'window_start']:
                if alt in df.columns:
                    df = df.rename(columns={alt: 'slot_start'})
                    break
        if len(df) == 0:
            self._slot_minutes = int(slot_minutes)
            self._map: Dict[Tuple[str, pd.Timestamp], float] = {}
            return
        df['slot_start'] = pd.to_datetime(df['slot_start'])
        df['zone_id'] = df['zone_id'].astype(str)
        df['pred_count'] = pd.to_numeric(df['pred_count'], errors='coerce').fillna(0.0)
        self._slot_minutes = int(slot_minutes)
        self._map: Dict[Tuple[str, pd.Timestamp], float] = {}
        for _, r in df.iterrows():
            key = (str(r['zone_id']), pd.Timestamp(r['slot_start']))
            self._map[key] = float(r['pred_count'])
    @property
    def slot_minutes(self) -> int:
        return self._slot_minutes
    def get(self, zone_id: str, slot_start: pd.Timestamp) -> float:
        return float(self._map.get((str(zone_id), pd.Timestamp(slot_start)), 0.0))

### 11.3 Depot hedging engine & helpers

In [ ]:
@dataclass
class HedgeKnobs:
    alpha_init: float = 1.5
    min_alpha: float = 1.0
    slot_minutes: int = 15
    topK_zones: int = 5
    decay_when_miss: float = 0.9
    eta_max_min: float = 45.0
    eps_km: float = 0.05

def _floor_to_slot(ts: pd.Timestamp, slot_minutes: int) -> pd.Timestamp:
     """Return the time ts to the beginning of the slot: 0', 15', 30', 45' according to slot_minutes."""
     ts = pd.to_datetime(ts)
     mins = (ts.minute // slot_minutes) * slot_minutes
     return ts.replace(minute=mins, second=0, microsecond=0)

def _arrival_slot(now: pd.Timestamp, eta_min: float, slot_minutes: int) -> tuple[pd.Timestamp, int]:
    """Calculate arrival time and corresponding slot + offset from current slot."""
    t_arr = pd.to_datetime(now) + pd.to_timedelta(eta_min, unit='m')
    s = _floor_to_slot(t_arr, slot_minutes)
    s0 = _floor_to_slot(now, slot_minutes)
    offset = int((t_arr - s0).total_seconds() // (slot_minutes * 60))
    return s, max(0, offset)

@dataclass
class HedgeState:
    active: bool = False

    anchor_A_lon: float = 0.0
    anchor_A_lat: float = 0.0
    AB_km: float = 0.0

    base_L_lon: float = 0.0
    base_L_lat: float = 0.0

    depot_lon: float = 0.0
    depot_lat: float = 0.0

    alpha_factor: float = 1.5
    alpha_budget_km: float = 0.0

    target_zone: Optional[str] = None
    target_lon: Optional[float] = None
    target_lat: Optional[float] = None
    target_is_depot: bool = False

    next_recalc_at: Optional[pd.Timestamp] = None
    last_pick_ts: Optional[pd.Timestamp] = None
    virtual_req_id: Optional[str] = None

@dataclass
class HedgeStats:
    hedge_steps: int = 0
    hedge_captures: int = 0
    hedge_reposition_km: float = 0.0
    virtual_log: list = field(default_factory=list)

class DepotHedgeEngine:
    def __init__(self, forecast_df: pd.DataFrame, zone_centroids_df: pd.DataFrame, knobs: HedgeKnobs = HedgeKnobs()):
        self.knobs = knobs
        self.forecast = ForecastAccessor(forecast_df, slot_minutes=knobs.slot_minutes)
        self.zones = ZoneCentroids(zone_centroids_df)
        self.stats = HedgeStats()


    def tick(self, now: pd.Timestamp, df: pd.DataFrame, taxis: Dict[str, object], speed_kmh: float, tick_s: int) -> None:
        for tx in taxis.values():
            if hasattr(tx, 'hedge') and tx.hedge.active and len(tx.route_open) > 0:
                try:
                    ridx0 = tx.route_open[0]
                    r = df.loc[ridx0]
                    px_lon = float(r['pickup_lon'])
                    px_lat = float(r['pickup_lat'])
                except Exception:
                    px_lon, px_lat = float(tx.cur_lon), float(tx.cur_lat)
                self.note_capture(tx, px_lon, px_lat, when=now)
                tx.hedge.active = False
                tx.hedge.target_zone = None
                tx.hedge.target_is_depot = False

        eligible: List[object] = []
        for tx in taxis.values():
            if not (tx.freeze_ptr == len(tx.route_freeze) and len(tx.route_open) == 0):
                continue
            eligible.append(tx)
        if not eligible:
            return

        step_km = max(0.0, float(speed_kmh)) * (float(tick_s) / 3600.0)
        for tx in eligible:
            st: HedgeState = getattr(tx, 'hedge', None)
            if st is None:
                st = HedgeState()
                setattr(tx, 'hedge', st)
            if not st.active:
                self._start_if_needed(tx)
                st = tx.hedge
                if not st.active:
                    continue
            if st.target_lon is None or st.target_lat is None:
                picked = self._pick_next_waypoint(tx, now, speed_kmh)
                if not picked:
                    self._fallback_to_depot_or_stop(tx, now)
                    st = tx.hedge
                    if not st.active:
                        continue
            moved_km, reached = self._advance_towards(tx, st.target_lon, st.target_lat, step_km)
            self.stats.hedge_reposition_km += moved_km

            tx.cur_time = now

            if reached and not st.target_is_depot:
                self.stats.hedge_steps += 1
                self._virt_close(tx, status="arrived_no_capture", when=now)
                st.base_L_lon, st.base_L_lat = tx.cur_lon, tx.cur_lat
                st.target_zone = None
                st.target_lon = None
                st.target_lat = None

                st.alpha_factor *= self.knobs.decay_when_miss

                if st.alpha_factor <= self.knobs.min_alpha:
                    self._fallback_to_depot_or_stop(tx, now)
                    continue

                st.alpha_budget_km = float(st.alpha_factor) * float(st.AB_km)

                picked2 = self._pick_next_waypoint(tx, now, speed_kmh)
                if not picked2:
                    self._fallback_to_depot_or_stop(tx, now)
                    st = tx.hedge
                    if not st.active:
                        continue

            elif reached and st.target_is_depot:
                self._virt_close(tx, status="closed_at_depot", when=now)
                st.active = False
                st.target_lon = None
                st.target_lat = None
                st.target_is_depot = False


    def _start_if_needed(self, tx: object) -> None:
        B_lon = getattr(tx, 'origin_lon', getattr(tx, 'cur_lon', 0.0))
        B_lat = getattr(tx, 'origin_lat', getattr(tx, 'cur_lat', 0.0))

        A_lon = float(getattr(tx, 'cur_lon', 0.0))
        A_lat = float(getattr(tx, 'cur_lat', 0.0))

        AB = haversine_km(A_lon, A_lat, B_lon, B_lat)

        st: Optional[HedgeState] = getattr(tx, 'hedge', None)
        if st is None:
            st = HedgeState()

        alpha = float(getattr(st, "alpha_factor", self.knobs.alpha_init))

        if alpha <= float(self.knobs.min_alpha):
          alpha = float(self.knobs.alpha_init)

        D = alpha * AB

        st.active = True

        st.anchor_A_lon = A_lon
        st.anchor_A_lat = A_lat
        st.AB_km = AB

        st.base_L_lon = A_lon
        st.base_L_lat = A_lat

        st.depot_lon = B_lon
        st.depot_lat = B_lat

        st.alpha_factor = alpha
        st.alpha_budget_km = D

        st.target_zone = None
        st.target_lon = None
        st.target_lat = None
        st.target_is_depot = False
        st.next_recalc_at = None
        st.last_pick_ts = None
        st.virtual_req_id = None

        setattr(tx, 'hedge', st)

    def _pick_next_waypoint(self, tx: object, now: pd.Timestamp, speed_kmh: float) -> bool:
        st: HedgeState = tx.hedge

        if st.alpha_factor <= self.knobs.min_alpha:
            return self._set_target_depot(tx, now)

        Llon, Llat = st.base_L_lon, st.base_L_lat
        Blon, Blat = st.depot_lon, st.depot_lat

        dist_LB = haversine_km(Llon, Llat, Blon, Blat)
        if dist_LB <= self.knobs.eps_km:
            return self._set_target_depot(tx, now)

        D = max(0.0, float(st.alpha_factor) * float(st.AB_km))
        st.alpha_budget_km = D

        if D <= dist_LB + 1e-9:
            return self._set_target_depot(tx, now)

        slot_m = self.knobs.slot_minutes
        candidates = []
        for z in self.zones.zone_ids:
            zlon, zlat = self.zones.centroid(z)

            km_LZ = haversine_km(Llon, Llat, zlon, zlat)
            km_ZB = haversine_km(zlon, zlat, Blon, Blat)

            if km_LZ + km_ZB > D + 1e-9:
                continue

            if speed_kmh <= 0:
                continue
            eta_min = (km_LZ / max(1e-9, speed_kmh)) * 60.0
            if self.knobs.eta_max_min > 0 and eta_min > self.knobs.eta_max_min:
                continue

            t_arr = now + pd.to_timedelta(eta_min, unit='m')
            s_arr = _floor_to_slot(t_arr, slot_m)
            demand = self.forecast.get(z, s_arr)

            extra = (km_LZ + km_ZB) - dist_LB
            candidates.append((z, demand, s_arr, extra, eta_min))

        if not candidates:
            return self._set_target_depot(tx, now)

        candidates.sort(key=lambda x: (x[2], -x[1], x[4]))
        candidates = candidates[:max(1, self.knobs.topK_zones)]

        z_pick, dem, s_arr, extra, eta_min = candidates[0]
        zlon, zlat = self.zones.centroid(z_pick)

        st.target_zone = z_pick
        st.target_lon = zlon
        st.target_lat = zlat
        st.target_is_depot = False
        st.last_pick_ts = now

        self._virt_open(tx, z_pick, now, eta_min, extra, D)
        return True

    def _fallback_to_depot_or_stop(self, tx: object, now: Optional[pd.Timestamp] = None) -> None:
        dist = haversine_km(tx.cur_lon, tx.cur_lat, tx.hedge.depot_lon, tx.hedge.depot_lat)

        if dist <= self.knobs.eps_km:
            self._virt_close(tx, status="closed_at_depot", when=(now or pd.Timestamp.utcnow()))
            tx.hedge.active = False
            tx.hedge.target_lon = None
            tx.hedge.target_lat = None
            tx.hedge.target_is_depot = False
        else:
            self._virt_close(tx, status="aborted_to_depot", when=(now or pd.Timestamp.utcnow()))
            self._set_target_depot(tx, now)

    def _set_target_depot(self, tx: object, now: Optional[pd.Timestamp]) -> bool:
        """Simple utility function, set the target of HedgeState to be the coordinates of the depot"""
        st: HedgeState = tx.hedge
        st.target_zone = None
        st.target_lon = st.depot_lon
        st.target_lat = st.depot_lat
        st.target_is_depot = True
        st.last_pick_ts = now
        return True

    @staticmethod
    def _advance_towards(tx: object, tgt_lon: float, tgt_lat: float, step_km: float) -> Tuple[float, bool]:
        cur_lon, cur_lat = float(tx.cur_lon), float(tx.cur_lat)
        d = haversine_km(cur_lon, cur_lat, tgt_lon, tgt_lat)

        if d <= 1e-9:
            return 0.0, True

        move = min(step_km, d)

        frac = 0.0 if d <= 0.0 else (move / d)

        new_lon = cur_lon + frac * (tgt_lon - cur_lon)
        new_lat = cur_lat + frac * (tgt_lat - cur_lat)

        tx.cur_lon = new_lon
        tx.cur_lat = new_lat

        reached = (d - move) <= 1e-6
        return move, reached


    def _virt_open(self, tx, z: str, now: pd.Timestamp, eta_min: float, extra_km: float, alpha_km: float):
        st: HedgeState = tx.hedge
        vid = str(uuid.uuid4())[:8]
        st.virtual_req_id = vid

        zlon, zlat = self.zones.centroid(z)
        entry = {
            "virt_id": vid,
            "taxi_id": getattr(tx, 'taxi_id', ''),
            "zone_id": z,
            "from_lon": st.base_L_lon,
            "from_lat": st.base_L_lat,
            "to_lon": zlon,
            "to_lat": zlat,
            "created_at": pd.Timestamp(now),
            "eta_min": float(eta_min),
            "extra_km": float(extra_km),
            "alpha_km": float(alpha_km),
            "status": "open",
        }
        self.stats.virtual_log.append(entry)

    def _virt_close(self, tx, status: str, when: pd.Timestamp, extra: Optional[dict] = None):
        st: HedgeState = getattr(tx, 'hedge', None)

        if st is None or not getattr(st, 'virtual_req_id', None):
            return

        vid = st.virtual_req_id

        for i in range(len(self.stats.virtual_log) - 1, -1, -1):
            row = self.stats.virtual_log[i]
            if row.get("virt_id") == vid and row.get("status") == "open":
                row["status"] = status
                row["closed_at"] = pd.Timestamp(when)
                if extra:
                    row.update(extra)
                break
        st.virtual_req_id = None

    def note_capture(self, tx, pickup_lon: float, pickup_lat: float, when: pd.Timestamp) -> None:
        best_z = None
        best_d = float('inf')

        for z in self.zones.zone_ids:
            zlon, zlat = self.zones.centroid(z)
            d = haversine_km(pickup_lon, pickup_lat, zlon, zlat)
            if d < best_d:
                best_d, best_z = d, z

        target = getattr(tx.hedge, 'target_zone', None)

        status = 'captured_in_target_zone' if (best_z is not None and target is not None and str(best_z) == str(target)) else 'captured_outside_target'

        try:
            st: HedgeState = tx.hedge
            if status == 'captured_in_target_zone':
                st.alpha_factor = float(self.knobs.alpha_init)
                st.alpha_budget_km = float(st.alpha_factor) * float(st.AB_km)
        except Exception:
            pass

        self._virt_close(
            tx,
            status=status,
            when=when,
            extra={
                'captured_lon': float(pickup_lon),
                'captured_lat': float(pickup_lat),
                'captured_zone_id': str(best_z) if best_z is not None else None,
                'target_zone_id': str(target) if target is not None else None,
            }
        )
        self.stats.hedge_captures += 1

## 12. Main loop

In [ ]:
def run_dynamic_no_horizon_v2(
    df_out: pd.DataFrame,
    fleet,
    *,
    tick_s: int = 60,
    freeze_window_s: int = 30,
    speed_kmh: float = 25.0,
    max_ops_per_tick: int = 50,
    candidate_taxi_limit: int = 100,
    pos_limit: int = 15,
    m_max_open: int = 15,
    log_every: int = 60,
    use_ejection: bool = True,
    eject_window: int = 4,
    forecast_df: Optional[pd.DataFrame] = None,
    zone_centroids_df: Optional[pd.DataFrame] = None,
    hedge_knobs: Optional[HedgeKnobs] = None,
) -> Tuple[pd.DataFrame, Dict, Dict[str, object], DepotHedgeEngine]:
    df = df_out.copy()

    for col in ["pickup_datetime", "dropoff_datetime", "pickup_earliest", "pickup_latest"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            try:
                df[col] = df[col].dt.tz_localize(None)
            except TypeError:
                pass

    sim_dtype = "datetime64[ns]"
    for c in ["sim_pickup_datetime", "sim_dropoff_datetime"]:
        if c not in df.columns:
            df[c] = pd.Series(pd.NaT, index=df.index, dtype=sim_dtype)
        else:
            df[c] = pd.to_datetime(df[c], errors="coerce")
            try:
                df[c] = df[c].dt.tz_localize(None)
            except TypeError:
                pass

    if 'assigned' not in df.columns:
        df['assigned'] = False
    else:
        df['assigned'] = df['assigned'].fillna(False)
    for c, default in [('assigned_taxi_id', None), ('assigned_depot_id', None)]:
        if c not in df.columns:
            df[c] = pd.Series([default]*len(df), dtype='object')
        else:
            df[c] = df[c].astype('object')
    if 'assigned_mode' not in df.columns:
        df['assigned_mode'] = ''
    else:
        df['assigned_mode'] = df['assigned_mode'].fillna('').astype('object')

    taxis = build_taxis_live(df, fleet)

    t_min = pd.to_datetime(df['pickup_earliest'] if 'pickup_earliest' in df.columns else df['pickup_datetime']).min()
    t_max = pd.to_datetime(df['pickup_latest'] if 'pickup_latest' in df.columns else df['pickup_datetime']).max()
    now = t_min
    end = t_max + pd.Timedelta(seconds=tick_s)

    stats = {'moves': MoveStats()}

    if forecast_df is None:
        forecast_df = pd.DataFrame(columns=['zone_id','slot_start','pred_count'])
    if zone_centroids_df is None:
        zone_centroids_df = pd.DataFrame(columns=['zone_id','longitude','latitude'])

    if hedge_knobs is None:
        hedge_knobs = HedgeKnobs()
    hedge = DepotHedgeEngine(forecast_df=forecast_df, zone_centroids_df=zone_centroids_df, knobs=hedge_knobs)

    last_now = now - pd.Timedelta(seconds=tick_s)
    ts_log = []
    while now <= end:
        _tick_t0 = time.perf_counter()

        update_freeze_open(df, taxis, now, freeze_window_s=freeze_window_s)
        advance_state_after_freeze(df, taxis, speed_kmh)

        removed = cap_open_lists(df, taxis, m_max_open)

        if removed:
            for ridx_rm in removed:
                ok = try_insert_best_with_load_bias(
                    df, taxis, ridx_rm, speed_kmh=speed_kmh,
                    candidate_taxi_limit=max(candidate_taxi_limit, 50),
                    pos_limit=pos_limit,
                    depot_bonus=0.5,
                    idle_bonus=0.3,
                    max_detour_ratio=1.5,
                )
                if (not ok) and use_ejection:
                    ok = assign_with_ejection(
                        df, taxis, ridx_rm, speed_kmh=speed_kmh,
                        candidate_taxi_limit=max(40, candidate_taxi_limit),
                        pos_limit=max(12, pos_limit),
                        eject_window=eject_window,
                    )

                if not ok:
                    df.at[ridx_rm, 'assigned'] = False
                    df.at[ridx_rm, 'assigned_mode'] = 'deferred_by_cap'

            cap_open_lists(df, taxis, m_max_open)

        new_ids = pull_new_requests(df, last_now, now)
        rej_this_tick = 0

        for ridx in new_ids:
            ok = preempt_if_better(
                df, taxis, ridx, speed_kmh=speed_kmh,
                candidate_taxi_limit=candidate_taxi_limit,
                preempt_positions=min(3, pos_limit),
            ) or try_insert_best_with_load_bias(
                df, taxis, ridx, speed_kmh=speed_kmh,
                candidate_taxi_limit=candidate_taxi_limit,
                pos_limit=pos_limit,
                depot_bonus=0.5,
                idle_bonus=0.3,
                max_detour_ratio=1.5
            )
            if (not ok) and use_ejection:
                ok = assign_with_ejection(
                    df, taxis, ridx, speed_kmh=speed_kmh,
                    candidate_taxi_limit=max(40, candidate_taxi_limit),
                    pos_limit=max(12, pos_limit),
                    eject_window=eject_window,
                )
            if not ok:
                px_l = df.at[ridx, 'pickup_latest'] if 'pickup_latest' in df.columns else df.at[ridx, 'pickup_datetime']
                if pd.notna(px_l) and now >= px_l:
                    df.at[ridx,'assigned'] = False
                    df.at[ridx,'assigned_mode'] = 'rejected'
                    rej_this_tick += 1

        ops = 0
        num_new = len(new_ids)
        while ops < max_ops_per_tick:
            props = [
                relocate_best_improvement(df, taxis, speed_kmh, candidate_taxi_limit=candidate_taxi_limit, pos_limit=pos_limit),
                swap_best_improvement(df, taxis, speed_kmh, candidate_taxi_limit=candidate_taxi_limit, pos_limit=pos_limit),
                cross_exchange_best_improvement(df, taxis, k=1, speed_kmh=speed_kmh, candidate_taxi_limit=candidate_taxi_limit, pos_limit=pos_limit),
            ]
            props = [p for p in props if p is not None]
            if not props:
                break

            best = min(props, key=lambda p: p.delta)
            ok_apply = apply_move(best, taxis)
            if not ok_apply: break

            stats['moves'].count[best.name] += 1
            stats['moves'].delta[best.name] += best.delta
            ops += 1

        hedge.tick(now, df, taxis, speed_kmh=speed_kmh, tick_s=tick_s)

        tick_idx = int((now - t_min).total_seconds() // tick_s)
        if log_every and (tick_idx % log_every == 0):
            _tick_dt = time.perf_counter() - _tick_t0
            freeze_cnt = sum(len(tx.route_freeze) for tx in taxis.values())
            open_cnt = sum(len(tx.route_open) for tx in taxis.values())
            print(f"[tick {now}] new={num_new}, rej={rej_this_tick}, ops={ops}, freeze={freeze_cnt}, open={open_cnt}, tick_time={_tick_dt:.3f}s")

        _elapsed_h = (now - t_min).total_seconds() / 3600.0
        _v = max(speed_kmh, 1e-9)

        _snap_fare = 0.0
        _snap_deadhead_km = 0.0
        _snap_total_km = 0.0
        _snap_cost_drive = 0.0
        _snap_cost_wait = 0.0

        _assigned_ridx = set()
        for _tx in taxis.values():
            _seq = _tx.route_freeze + _tx.route_open
            _tx_total_km = route_cost_km_from_origin(df, _seq, _tx.origin_lon, _tx.origin_lat) if _seq else 0.0
            _tx_dead_km  = deadhead_km_from_origin(df, _seq, _tx.origin_lon, _tx.origin_lat)   if _seq else 0.0
            _tx_drive_h  = _tx_total_km / _v
            _snap_cost_drive += COST_PER_HOUR_DRIVE * _tx_drive_h
            _idle_h = max(0.0, _elapsed_h - _tx_drive_h)
            _snap_cost_wait += COST_PER_HOUR_WAIT * _idle_h
            _snap_total_km    += _tx_total_km
            _snap_deadhead_km += _tx_dead_km
            for _ridx in _seq: _assigned_ridx.add(_ridx)

        _snap_trip_km = _snap_total_km - _snap_deadhead_km

        if _assigned_ridx:
            for _ridx in _assigned_ridx:
                _row = df.iloc[_ridx]
                _km_trip_r = haversine_km(_row.get('pickup_lon', 0.0), _row.get('pickup_lat', 0.0), _row.get('dropoff_lon', 0.0), _row.get('dropoff_lat', 0.0))
                _snap_fare += FARE_PER_HOUR_DRIVE * (_km_trip_r / _v)

        _snap_profit = _snap_fare - _snap_cost_drive - _snap_cost_wait
        _snap_served = int((df['assigned'] == True).sum())
        _appeared = int((df['pickup_datetime'] <= now).sum()) if 'pickup_datetime' in df.columns else len(df)
        _snap_rejected = _appeared - _snap_served

        ts_log.append({
            'time':           (now - t_min).total_seconds(),
            'elapsed_min':    (now - t_min).total_seconds() / 60.0,
            'cum_fare':       round(_snap_fare, 4),
            'cum_profit':     round(_snap_profit, 4),
            'cum_deadhead_km': round(_snap_deadhead_km, 4),
            'cum_total_km':   round(_snap_total_km, 4),
            'cum_trip_km':    round(_snap_trip_km, 4),
            'cum_cost_drive': round(_snap_cost_drive, 4),
            'cum_cost_wait':  round(_snap_cost_wait, 4),
            'cum_served':     _snap_served,
            'cum_rejected':   _snap_rejected,
        })

        last_now = now
        now += pd.Timedelta(seconds=tick_s)

    for tx in taxis.values():
        for ridx in tx.route_freeze + tx.route_open:
            df.at[ridx, 'assigned'] = True
            df.at[ridx, 'assigned_taxi_id'] = tx.taxi_id
            df.at[ridx, 'assigned_depot_id'] = tx.depot_id

    materialize_sim_times(df, taxis, speed_kmh)
    deadline = end
    mask_to_close = ((df['assigned'] == False) & (pd.to_datetime(df['pickup_latest'] if 'pickup_latest' in df.columns else df['pickup_datetime']) <= deadline))
    df.loc[mask_to_close, 'assigned_mode'] = 'rejected'

    df, profit_by_taxi_paper, system_profit_paper, system_fare_paper, system_cost_drive_paper, system_cost_wait_paper, system_cost_deadhead_paper, system_cost_loaded_paper = compute_profit_paper_style(df, taxis, speed_kmh=speed_kmh)
    system_fare_paper = sum(v["total_fare_paper"] for v in profit_by_taxi_paper.values())
    system_cost_drive_paper = sum(v["total_cost_drive_paper"] for v in profit_by_taxi_paper.values())

    _sim_duration_h = (end - t_min).total_seconds() / 3600.0
    _sum_drive_km = 0.0
    for _tid, _tx in taxis.items():
        _seq = _tx.route_freeze + _tx.route_open
        if _seq: _sum_drive_km += route_cost_km_from_origin(df, _seq, _tx.origin_lon, _tx.origin_lat)
    _total_drive_h_summary = _sum_drive_km / max(speed_kmh, 1e-9)
    _fleet_idle_h_summary = max(0.0, len(taxis) * _sim_duration_h - _total_drive_h_summary)
    system_cost_wait_paper = COST_PER_HOUR_WAIT * _fleet_idle_h_summary
    system_profit_paper = system_fare_paper - system_cost_drive_paper - system_cost_wait_paper

    used_taxis = set(df.loc[df["assigned"] == True, "assigned_taxi_id"].dropna().astype(str).tolist())
    total_taxis = len(taxis)
    used_cnt = len(used_taxis)

    total_km, dead_km = 0.0, 0.0
    per_taxi_km = {}
    for tid in used_taxis:
        tx = taxis[tid]
        seq = tx.route_freeze + tx.route_open
        km_total = route_cost_km_from_origin(df, seq, tx.origin_lon, tx.origin_lat)
        km_dead  = deadhead_km_from_origin(df, seq, tx.origin_lon, tx.origin_lat)
        total_km += km_total
        dead_km  += km_dead
        per_taxi_km[tid] = {"total_km": km_total, "deadhead_km": km_dead}

    if per_taxi_km:
        total_vals = np.array([v["total_km" ] for v in per_taxi_km.values()])
        dead_vals  = np.array([v["deadhead_km"] for v in per_taxi_km.values()])
        taxi_total_km_max, taxi_total_km_min, taxi_total_km_avg = float(total_vals.max()), float(total_vals.min()), float(total_vals.mean())
        taxi_dead_km_max, taxi_dead_km_min, taxi_dead_km_avg = float(dead_vals.max()), float(dead_vals.min()), float(dead_vals.mean())
    else:
        taxi_total_km_max = taxi_total_km_min = taxi_total_km_avg = 0.0
        taxi_dead_km_max = taxi_dead_km_min = taxi_dead_km_avg = 0.0

    if 'km_total_paper' in df.columns:
        served_mask = (df['assigned'] == True) & df['sim_pickup_datetime'].notna()
        if served_mask.any():
            req_km = df.loc[served_mask, 'km_total_paper'].to_numpy(dtype=float)
            request_km_max, request_km_min = float(req_km.max()), float(req_km.min())
        else: request_km_max = request_km_min = 0.0
    else: request_km_max = request_km_min = 0.0

    if len(hedge.stats.virtual_log) > 0:
        _vdf = pd.DataFrame(hedge.stats.virtual_log)
        _statuses = _vdf['status']
        _hedge_arrived_nc = int((_statuses == 'arrived_no_capture').sum())
        _hedge_cap_in = int((_statuses == 'captured_in_target_zone').sum())
        _hedge_cap_out = int((_statuses == 'captured_outside_target').sum())
    else: _hedge_arrived_nc = _hedge_cap_in = _hedge_cap_out = 0
    _hedge_cap_total = _hedge_cap_in + _hedge_cap_out

    report = {
        "num_taxis_used": f"{used_cnt}/{total_taxis}",
        'total_km': total_km, 'served': int((df['assigned'] == True).sum()), 'rejected': int((df.get('assigned_mode','') == 'rejected').sum()), 'deadhead_km': dead_km,
        'system_fare_paper': system_fare_paper, 'system_cost_drive_paper': system_cost_drive_paper, 'system_cost_deadhead_paper': system_cost_deadhead_paper,
        'system_cost_loaded_paper': system_cost_loaded_paper, 'system_cost_wait_paper': system_cost_wait_paper, 'system_profit_paper': system_profit_paper,
        'taxi_total_km_max': taxi_total_km_max, 'taxi_total_km_min': taxi_total_km_min, 'taxi_total_km_avg': taxi_total_km_avg,
        'taxi_deadhead_km_max': taxi_dead_km_max, 'taxi_deadhead_km_min': taxi_dead_km_min, 'taxi_deadhead_km_avg': taxi_dead_km_avg,
        'request_km_max': request_km_max, 'request_km_min': request_km_min,
        'hedge_reposition_km': hedge.stats.hedge_reposition_km, 'hedge_steps': hedge.stats.hedge_steps, 'hedge_captures_total': _hedge_cap_total,
        'hedge_captures_in_target_zone': _hedge_cap_in, 'hedge_captures_outside_target': _hedge_cap_out, 'hedge_arrivals_no_capture': _hedge_arrived_nc,
        'move_counts': stats['moves'].count, 'move_effects_km': stats['moves'].delta,
    }

    ts_df = pd.DataFrame(ts_log)
    report['time_series'] = ts_df
    return df, report, taxis, hedge

## 13. Functions that support loading data

### 13.0 Data load

In [ ]:
def load_forecast_15m_auto(
    file_rel_path: str, *, data_path: Path = DATA_PATH, slot_minutes: int = 15
) -> pd.DataFrame:
    """
    Flexible loader for 15-minute forecast file:
    - WIDE format support:
        t_base, t_plus_15min, Zone 0, Zone 1, ...
      or: t_base, Zone0, Zone1, ...
    - If there are many dates in one file, it's still OK: just t_base is the full timestamp.
    - Standardize output: zone_id (Zone0..ZoneK), slot_start, pred_count.
    """
    fp = data_path / file_rel_path
    f = pd.read_csv(fp)

    zone_cols = []
    for c in f.columns:
        s = str(c)
        if s.startswith("Zone"):
            tail = s[4:].strip()
            if tail.isdigit():
                zone_cols.append(c)

    has_base = "t_base" in f.columns
    has_plus = "t_plus_15min" in f.columns

    if (has_base or has_plus) and len(zone_cols) > 0:
        if has_base:
            slot = pd.to_datetime(f["t_base"], errors="coerce", format="mixed")
        else:
            slot = pd.to_datetime(f["t_plus_15min"], errors="coerce", format="mixed")

        f["slot_start"] = slot.dt.floor(f"{int(slot_minutes)}min")

        for zc in zone_cols:
            f[zc] = pd.to_numeric(f[zc], errors="coerce").fillna(0.0)

        long = f.melt(
            id_vars=["slot_start"],
            value_vars=zone_cols,
            var_name="zone_id",
            value_name="pred_count",
        )

        long["zone_id"] = (
            long["zone_id"]
            .astype(str)
            .str.replace(" ", "", regex=False)
        )

        out = long.groupby(["zone_id", "slot_start"], as_index=False)["pred_count"].sum()
        return out

    if "zone_id" not in f.columns and "zone" in f.columns:
        f = f.rename(columns={"zone": "zone_id"})

    if "pred_count" not in f.columns:
        for alt in ["pred", "count", "yhat", "forecast", "value"]:
            if alt in f.columns:
                f = f.rename(columns={alt: "pred_count"})
                break

    if "slot_start" not in f.columns:
        for alt in ["slot", "time", "ts", "window_start", "t_base", "t_plus_15min"]:
            if alt in f.columns:
                f = f.rename(columns={alt: "slot_start"})
                break

    f["zone_id"] = f["zone_id"].astype(str)
    f["slot_start"] = pd.to_datetime(f["slot_start"], errors="coerce", format="mixed").dt.floor(
        f"{int(slot_minutes)}min"
    )
    f["pred_count"] = pd.to_numeric(f["pred_count"], errors="coerce").fillna(0.0)

    f = f.groupby(["zone_id", "slot_start"], as_index=False)["pred_count"].sum()
    return f


def _coerce_lon_lat_cols(df: pd.DataFrame, lon_keys=("longitude","lon","long","x"), lat_keys=("latitude","lat","y")):
    out = df.copy()
    if 'longitude' not in out.columns:
        for k in lon_keys:
            if k in out.columns:
                out = out.rename(columns={k:'longitude'})
                break
    if 'latitude' not in out.columns:
        for k in lat_keys:
            if k in out.columns:
                out = out.rename(columns={k:'latitude'})
                break
    return out


def _scale_if_needed(df: pd.DataFrame, taxi_factor: Optional[float], rounding: str = 'ceil') -> pd.DataFrame:
    if not taxi_factor or taxi_factor == 1.0:
        return df
    try:
        return scale_taxis_per_depot(df, factor=taxi_factor, rounding=rounding)
    except Exception:
        out = df.copy()
        if 'taxis_per_depot' not in out.columns:
            out['taxis_per_depot'] = 1
        x = pd.to_numeric(out['taxis_per_depot'], errors='coerce').fillna(0)
        if rounding == 'ceil':
            out['taxis_per_depot'] = np.ceil(x * taxi_factor).astype(int)
        elif rounding == 'round':
            out['taxis_per_depot'] = np.rint(x * taxi_factor).astype(int)
        else:
            out['taxis_per_depot'] = np.floor(x * taxi_factor).astype(int)
        out['taxis_per_depot'] = out['taxis_per_depot'].clip(lower=0)
        return out


def load_requests_day(file_rel_path: str,
                      *,
                      data_path: Path = DATA_PATH,
                      taxi_factor: Optional[float] = None,
                      rounding: str = 'ceil',
                      early_s: int = 0,
                      late_s: int = 360) -> pd.DataFrame:
    fp = data_path / file_rel_path
    df = pd.read_csv(fp, low_memory=False)

    try:
        df = normalize_requests_df(df)
    except Exception:
        out = df.copy()
        for c in ('pickup_datetime','dropoff_datetime'):
            if c in out.columns:
                out[c] = pd.to_datetime(out[c], errors='coerce')
        if 'pickup_lon' not in out.columns and 'pickup_longitude' in out.columns:
            out['pickup_lon'] = out['pickup_longitude']
        if 'pickup_lat' not in out.columns and 'pickup_latitude' in out.columns:
            out['pickup_lat'] = out['pickup_latitude']
        if 'dropoff_lon' not in out.columns and 'dropoff_longitude' in out.columns:
            out['dropoff_lon'] = out['dropoff_longitude']
        if 'dropoff_lat' not in out.columns and 'dropoff_latitude' in out.columns:
            out['dropoff_lat'] = out['dropoff_latitude']
        for name, default in [('assigned', False), ('assigned_taxi_id', None), ('assigned_depot_id', None), ('assigned_mode','')]:
            if name not in out.columns:
                out[name] = default
        df = out

    try:
        df = ensure_pickup_windows(df, early_s=early_s, late_s=late_s)
    except Exception:
        if 'pickup_earliest' not in df.columns:
            df['pickup_earliest'] = pd.to_datetime(df['pickup_datetime']) - pd.to_timedelta(early_s, unit='s')
        if 'pickup_latest' not in df.columns:
            df['pickup_latest']  = pd.to_datetime(df['pickup_datetime']) + pd.to_timedelta(late_s, unit='s')

    df = _scale_if_needed(df, taxi_factor, rounding)

    df = df.sort_values('pickup_earliest').reset_index(drop=True)
    return df


def load_zone_centroids(file_rel_path: str, *, data_path: Path = DATA_PATH, zone_prefix: str = 'Zone') -> pd.DataFrame:
    fp = data_path / file_rel_path
    z = pd.read_csv(fp)
    z = _coerce_lon_lat_cols(z)
    if 'zone_id' not in z.columns:
        z = z.reset_index(drop=True)
        z['zone_id'] = [f"{zone_prefix}{i}" for i in range(len(z))]
    else:
        z['zone_id'] = z['zone_id'].astype(str)
    return z[['zone_id','longitude','latitude']]


def load_forecast_15m(file_rel_path: str, *, data_path: Path = DATA_PATH, slot_minutes: int = 15) -> pd.DataFrame:
    fp = data_path / file_rel_path
    f = pd.read_csv(fp)
    if 'zone_id' not in f.columns and 'zone' in f.columns:
        f = f.rename(columns={'zone':'zone_id'})
    if 'pred_count' not in f.columns:
        for alt in ['pred','count','yhat','forecast','value']:
            if alt in f.columns:
                f = f.rename(columns={alt:'pred_count'})
                break
    if 'slot_start' not in f.columns:
        for alt in ['slot','time','ts','window_start']:
            if alt in f.columns:
                f = f.rename(columns={alt:'slot_start'})
                break
    f['zone_id'] = f['zone_id'].astype(str)
    f['slot_start'] = pd.to_datetime(f['slot_start'], errors='coerce')
    f['slot_start'] = f['slot_start'].dt.floor(f'{int(slot_minutes)}min')
    f['pred_count'] = pd.to_numeric(f['pred_count'], errors='coerce').fillna(0.0)
    f = (f.groupby(['zone_id','slot_start'], as_index=False)['pred_count'].sum())
    return f


def prepare_inputs(requests_rel: str,
                   centroids_rel: str,
                   forecast_rel: Optional[str] = None,
                   *,
                   data_path: Path = DATA_PATH,
                   taxi_factor: Optional[float] = None,
                   rounding: str = 'ceil',
                   early_s: int = 0,
                   late_s: int = 360):
    req = load_requests_day(requests_rel, data_path=data_path, taxi_factor=taxi_factor,
                            rounding=rounding, early_s=early_s, late_s=late_s)
    centroids_df = load_zone_centroids(centroids_rel, data_path=data_path)
    forecast_df = (load_forecast_15m_auto(forecast_rel, data_path=data_path)
                   if forecast_rel else pd.DataFrame(columns=['zone_id','slot_start','pred_count']))

    fleet = Fleet.from_depots(req)
    return req, fleet, centroids_df, forecast_df

### 13.1 Run the simulation

# Function that outputs results

In [ ]:
PREFERRED_ORDER = [
    "datetime","num_taxis_used","served","rejected",
    "total_km","deadhead_km",
    "system_fare_paper","system_cost_drive_paper",
    "system_cost_deadhead_paper","system_cost_loaded_paper",
    "system_cost_wait_paper","system_profit_paper",
    "taxi_total_km_max","taxi_total_km_min","taxi_total_km_avg",
    "taxi_deadhead_km_max","taxi_deadhead_km_min","taxi_deadhead_km_avg",
    "request_km_max","request_km_min",
    "hedge_reposition_km","hedge_steps","hedge_captures","hedge_captures_total",
    "hedge_captures_in_target_zone","hedge_captures_outside_target","hedge_arrivals_no_capture",
    "move_counts_relocate","move_counts_swap","move_counts_cross",
    "move_effects_km_relocate","move_effects_km_swap","move_effects_km_cross",
]


def _flatten_report(report: dict) -> dict:
    flat = {}
    for k, v in report.items():
        if k in ("time_series", "profit_by_taxi_paper"):
            continue
        if isinstance(v, dict):
            for kk, vv in v.items():
                flat[f"{k}_{kk}"] = vv
        else:
            flat[k] = v
    return flat

def save_report_to_csv(report: dict, date_value: str, csv_path: str) -> str:
    """Append a line of results to the CSV (create a file if you don't already have one)."""
    row = {"datetime": date_value}
    row.update(_flatten_report(report))

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        for key in row.keys():
            if key not in df.columns:
                df[key] = pd.NA
        cols = list(df.columns)
        new_row_df = pd.DataFrame([row], columns=cols)
        df = pd.concat([df, new_row_df], ignore_index=True)
        df.to_csv(csv_path, index=False)
    else:
        extra_cols = [k for k in row.keys() if k not in PREFERRED_ORDER]
        cols = PREFERRED_ORDER + [k for k in extra_cols if k not in PREFERRED_ORDER]
        pd.DataFrame([row], columns=cols).to_csv(csv_path, index=False)
    return csv_path


# Run the dynamic function

In [ ]:
try:
    except Exception:
    pass

days = [
     "2016-06-14",
]

requests_tpl  = "last_month_daily/last_month_daily_eff_3/requests_{}.csv"
centroids_rel = "clusters/n20_init20/centroids_k20_n20.csv"
forecast_tpl  = "BASELINES/xgboost/xgboost_next15min_predictions.csv"

out_csv_path = r"d:/DTRP-DP/Routing/Output/dynamic_v2_outputs/ket_qua_cac_chi_so/xgb/v2_xgb.csv"

all_virt_logs = []

for date_str in days:
    print("\n==============================")
    print(f"RUNNING DATE: {date_str}")
    print("==============================")

    requests_rel = requests_tpl.format(date_str)
    forecast_rel = forecast_tpl.format(date_str)

    req, fleet_sim, centroids_df, forecast_df = prepare_inputs(
        requests_rel=requests_rel,
        centroids_rel=centroids_rel,
        forecast_rel=forecast_rel,
        data_path=DATA_PATH,
        taxi_factor=None,
        early_s=0,
        late_s=360,
    )

    df2, report2, taxis2, hedge = run_dynamic_no_horizon_v2(
        df_out=req, fleet=fleet_sim,
        tick_s=60, freeze_window_s=30, speed_kmh=25.0,
        candidate_taxi_limit=120, pos_limit=16, m_max_open=15,
        forecast_df=forecast_df,
        zone_centroids_df=centroids_df
    )

    print("== Report (", date_str, ") ==")
    for k, v in report2.items():
        if k not in ("profit_by_taxi_paper", "time_series"):
            print(f"{k}: {v}")

    virt_log = pd.DataFrame(hedge.stats.virtual_log)
    if not virt_log.empty:
        virt_log["date"] = date_str
        all_virt_logs.append(virt_log)
        print("\n== Virtual log sample (", date_str, ") ==")
        print(virt_log.head(5))
    else:
        print("\n== There is no virtual log for the day", date_str, "==")

    csv_out = save_report_to_csv(
        report2,
        date_str,
        out_csv_path
    )

    print("Saved summary to:", csv_out)

    ts_df = report2.get('time_series', pd.DataFrame())
    if not ts_df.empty:
        ts_dir = Path(out_csv_path).parent / "time_series"
        ts_dir.mkdir(parents=True, exist_ok=True)
        ts_csv_path = ts_dir / f"ts_{date_str}.csv"
        ts_df.to_csv(ts_csv_path, index=False)
        print(f"Saved time-series to: {ts_csv_path}")

Mounted at /content/drive

ĐANG CHẠY NGÀY: 2016-06-14
[tick 2016-06-14 00:00:11] new=1, rej=0, ops=0, freeze=0, open=1, tick_time=0.063s
[tick 2016-06-14 01:00:11] new=5, rej=0, ops=0, freeze=196, open=3, tick_time=0.476s
[tick 2016-06-14 02:00:11] new=3, rej=0, ops=0, freeze=283, open=2, tick_time=0.249s
[tick 2016-06-14 03:00:11] new=1, rej=0, ops=0, freeze=340, open=1, tick_time=0.106s
[tick 2016-06-14 04:00:11] new=0, rej=0, ops=0, freeze=386, open=0, tick_time=0.005s
[tick 2016-06-14 05:00:11] new=0, rej=0, ops=0, freeze=418, open=0, tick_time=0.007s
[tick 2016-06-14 06:00:11] new=7, rej=0, ops=0, freeze=468, open=4, tick_time=0.627s
[tick 2016-06-14 07:00:11] new=6, rej=0, ops=1, freeze=681, open=5, tick_time=1.056s
[tick 2016-06-14 08:00:11] new=8, rej=0, ops=1, freeze=978, open=5, tick_time=0.844s
[tick 2016-06-14 09:00:11] new=8, rej=0, ops=1, freeze=1356, open=8, tick_time=1.200s
[tick 2016-06-14 10:00:11] new=7, rej=0, ops=0, freeze=1726, open=7, tick_time=0.661s
[tick 2016-